# 02 — Prompt Synthesis

Generates text prompts for all four experimental conditions using Claude as the synthesis LLM.
Prompts are optimised for patch-level CLIP cosine similarity localisation.

**Conditions:**
- `flat` (A): one-line concept label — baseline
- `expanded` (B): LLM-expanded without graph — controls for prompt length
- `graphrag` (C): full graph (spatial + non-spatial edges) → LLM
- `graphrag_spatial` (D): spatial edges only → LLM — spatially-weighted GraphRAG

**Requires:** `$DRIVE_BASE/knowledge_graphs/pathology_graphs.json` from notebook 01.

**Requires:** `ANTHROPIC_API_KEY` set in Colab secrets (🔑 icon in sidebar).

**Output:** `$DRIVE_BASE/prompts/all_prompts.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install anthropic -q

import os, json
from google.colab import userdata
import anthropic

DRIVE_BASE  = '/content/drive/MyDrive/medvit-graphrag'
PROMPTS_DIR = f'{DRIVE_BASE}/prompts'
os.makedirs(PROMPTS_DIR, exist_ok=True)

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

with open(f'{DRIVE_BASE}/knowledge_graphs/pathology_graphs.json') as f:
    graphs = json.load(f)

print(f'Graphs loaded: {list(graphs.keys())}')
print('Client initialised.')

In [ ]:
PATHOLOGIES = ['Pleural Effusion', 'Cardiomegaly', 'Edema', 'Lung Opacity']

FLAT_PROMPTS = {
    'Pleural Effusion': 'pleural effusion on chest X-ray',
    'Cardiomegaly':     'cardiomegaly on chest X-ray',
    'Edema':            'pulmonary edema on chest X-ray',
    'Lung Opacity':     'lung opacity on chest X-ray',
}

# System prompt engineered for CLIP-optimised output:
# dense noun phrases, location-first, no diagnostic inference, no hedging
SYSTEM_PROMPT = """You are writing text prompts for a CLIP-style vision-language model \
that localises chest X-ray findings by computing patch-level image-text cosine similarity.

Rules:
- Write exactly 1-2 sentences, no more.
- Use dense visual and spatial noun phrases — describe what the finding LOOKS LIKE and WHERE it appears.
- Prioritise anatomical location (e.g. "lower lung zones", "perihilar region", "cardiac silhouette").
- Prioritise visual appearance (e.g. "homogeneous opacity", "blunting", "enlarged shadow").
- Do NOT mention causes, patient history, symptoms, treatment, or diagnostic inference.
- Do NOT use hedging language ("may", "can", "suggests").
- Output only the prompt text, no preamble."""

In [ ]:
def get_spatial_subgraph(pathology: str, graphs: dict) -> dict:
    """
    Returns the spatially-weighted subgraph for a pathology.
    Keeps all anatomy nodes (always spatially grounding) plus
    observation nodes that appear in at least one spatial edge.
    """
    graph         = graphs[pathology]
    spatial_edges = [e for e in graph['edges'] if e[3]]
    spatial_obs   = set()
    for e in spatial_edges:
        spatial_obs.add(e[0])
        spatial_obs.add(e[2])
    kept_nodes = {
        k: v for k, v in graph['nodes'].items()
        if v['type'] == 'Anatomy' or k in spatial_obs
    }
    kept_edges = [
        e for e in spatial_edges
        if e[0] in kept_nodes and e[2] in kept_nodes
    ]
    return {'nodes': kept_nodes, 'edges': kept_edges}


def subgraph_to_triplets(pathology: str, graph: dict, spatial_only: bool) -> str:
    """Formats graph as readable triplet list for LLM input."""
    edges = [e for e in graph['edges'] if (not spatial_only or e[3])]
    lines = [f'Target finding: {pathology}', '', 'Knowledge graph triplets:']
    for e in edges:
        src = graph['nodes'].get(e[0], {}).get('label', e[0])
        rel = e[1].replace('_', ' ')
        tgt = graph['nodes'].get(e[2], {}).get('label', e[2])
        lines.append(f'  ({src}) --[{rel}]--> ({tgt})')
    return '\n'.join(lines)


def synthesise_prompt(pathology: str, triplets: str) -> str:
    """Calls Claude to synthesise a localisation prompt from a subgraph."""
    user = (
        f'Using the following clinical knowledge triplets, write a spatially grounded '
        f'visual description of {pathology} as seen on a chest X-ray:\n\n'
        f'{triplets}\n\nPrompt:'
    )
    response = client.messages.create(
        model='claude-opus-4-5',
        max_tokens=150,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user}]
    )
    return response.content[0].text.strip()


def synthesise_expanded_from_flat(pathology: str) -> str:
    """
    Condition B: LLM expands the flat prompt with no graph.
    Controls for prompt richness without graph structure.
    """
    user = (
        f'Expand the following chest X-ray finding name into a richer visual description '
        f'for use as a CLIP localisation prompt. Use only visual and spatial features. '
        f'Write 1-2 sentences.\n\nFinding: {pathology}\n\nPrompt:'
    )
    response = client.messages.create(
        model='claude-opus-4-5',
        max_tokens=150,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user}]
    )
    return response.content[0].text.strip()

In [ ]:
print('Generating prompts for all conditions...\n')

EXPANDED_PROMPTS         = {}
GRAPHRAG_PROMPTS         = {}
GRAPHRAG_SPATIAL_PROMPTS = {}

for pathology in PATHOLOGIES:
    print(f'  {pathology}...')

    EXPANDED_PROMPTS[pathology] = synthesise_expanded_from_flat(pathology)

    full_triplets = subgraph_to_triplets(pathology, graphs[pathology], spatial_only=False)
    GRAPHRAG_PROMPTS[pathology] = synthesise_prompt(pathology, full_triplets)

    spatial_graph    = get_spatial_subgraph(pathology, graphs)
    spatial_triplets = subgraph_to_triplets(pathology, spatial_graph, spatial_only=True)
    GRAPHRAG_SPATIAL_PROMPTS[pathology] = synthesise_prompt(pathology, spatial_triplets)

ALL_PROMPTS = {
    'flat':             FLAT_PROMPTS,
    'expanded':         EXPANDED_PROMPTS,
    'graphrag':         GRAPHRAG_PROMPTS,
    'graphrag_spatial': GRAPHRAG_SPATIAL_PROMPTS,
}

out_path = f'{PROMPTS_DIR}/all_prompts.json'
with open(out_path, 'w') as f:
    json.dump(ALL_PROMPTS, f, indent=2)

print(f'\nSaved to {out_path}')
print('\nGenerated prompts:\n')
for condition, prompts in ALL_PROMPTS.items():
    print(f'=== {condition.upper()} ===')
    for pathology, prompt in prompts.items():
        print(f'  [{pathology}]\n  {prompt}\n')